In [2]:
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Dense, Input, Dropout, Attention
from tensorflow.keras.models import Model

import os

import sagemaker
from sagemaker.tensorflow import TensorFlowModel
import boto3

import numpy as np

import time

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


In [3]:
def  model_recovery(ncols):
        # Définition du modèle
        inputs = Input(shape=(ncols,1))
        # First - layer - LSTM
        first_layer = LSTM(50, activation='relu', return_sequences=True)(inputs)
        # Second layer
        second_layer  = Dropout(0.2)(first_layer)
        # Third layer - Attention
        attention_out = Attention()([second_layer, second_layer])
        pooled = tf.keras.layers.GlobalAveragePooling1D()(attention_out)
        # Fourth layer - output layer
        outputs = Dense(1, activation='relu')(pooled)
        ml_model = Model(inputs, outputs)
        ml_model.compile(optimizer='adam', loss='mean_squared_error')  
        return ml_model

In [4]:
model_path = os.path.join("custom_attention.weights.h5")
ncols = 7
model = model_recovery(ncols)
model.load_weights(model_path)

In [5]:
model.export("model/1")

Saved artifact at 'model/1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 7, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140574231490448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140574231492560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140574231490064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140574231491792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140574231490256: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [6]:
#!tar -czvf model.tar.gz pretrained_model/
!tar -czvf model.tar.gz model/1/

model/1/
model/1/assets/
model/1/variables/
model/1/variables/variables.data-00000-of-00001
model/1/variables/variables.index
model/1/fingerprint.pb
model/1/saved_model.pb


In [7]:
import boto3
sagemaker_session = sagemaker.Session()
prefix = "finance_data"
training_input_path = sagemaker_session.upload_data('model.tar.gz', key_prefix=prefix + '/trained_model')

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


In [8]:
training_input_path

's3://amazon-sagemaker-102268067378-eu-north-1-cf61tgrhand6s3/shared/finance_data/trained_model/model.tar.gz'

In [9]:
tf_version = tf.__version__
role = sagemaker.get_execution_role()
sagemaker_session = sagemaker.Session()

tf_model = TensorFlowModel(
    model_data=training_input_path,
    role=role,
    framework_version=tf_version,  
    sagemaker_session=sagemaker_session
)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


In [10]:
#!pip install --upgrade sagemaker

In [12]:
tf_endpoint_name = 'tf-iris-model'+time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())
tf_predictor = tf_model.deploy(initial_instance_count=1,
                                   instance_type="ml.m5.large",#'ml.c5.large',
                                   py_version='py310',
                                   endpoint_name=tf_endpoint_name)

----!

In [13]:
#sample = np.array([0.1,0.2,0.3,0.4, 0.5,0.6,0.7]).to_list()
#sample = sample.reshape((ncols, 1))
sample = np.random.rand(1,7, 1)#.tolist()
response = tf_predictor.predict({"instances": sample})
print(response)

{'predictions': [[81.992485]]}


In [14]:
tf_predictor.delete_endpoint()